# Solution 01: Generate NER Dataset with LLM

In [ ]:
import re, json, os
from openai import OpenAI
from collections import Counter
from tqdm import tqdm

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "raw_security_reports.json")) as f:
    reports = json.load(f)

api_key = "your-api-key-here"  # Replace with your actual OpenAI key
client = OpenAI(api_key=api_key)

ENTITY_TYPES = ["malware", "vulnerability", "attack_technique", "affected_software", "threat_actor"]

def tokenize(text):
    return re.findall(r'\w+(?:[-_]\w+)*|\S', text)

print(f"✓ Loaded {len(reports)} reports")

## Part 1: Call LLM and Parse Response

In [2]:
SYSTEM_PROMPT = """Extract named entities from cybersecurity texts.
Output one entity per line in format: entity_text <> entity_type
Use ONLY these entity types: malware, vulnerability, attack_technique, affected_software, threat_actor
Output ONLY entity lines, nothing else. Do not include explanations or headers."""


def call_llm_ner(client, text, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text}
        ],
        temperature=0
    )
    return response.choices[0].message.content


def parse_ner_output(raw_output):
    entities = []
    for line in raw_output.strip().split('\n'):
        line = line.strip()
        if '<>' not in line:
            continue
        parts = line.split('<>')
        if len(parts) == 2:
            entity_text = parts[0].strip()
            entity_type = parts[1].strip().lower()
            if entity_text and entity_type in ENTITY_TYPES:
                entities.append((entity_text, entity_type))
    return entities


test_text = reports[0]['text']
raw_output = call_llm_ner(client, test_text)
entities = parse_ner_output(raw_output)

assert len(entities) >= 2
for e, t in entities:
    assert t in ENTITY_TYPES
print(f"✓ Parsed {len(entities)} entities from report 0")
for e_text, e_type in entities:
    print(f"  [{e_type}] '{e_text}'")

✓ Parsed 5 entities from report 0
  [threat_actor] 'LockBit 3.0'
  [vulnerability] 'CVE-2023-4966'
  [affected_software] 'Citrix NetScaler ADC'
  [affected_software] 'Windows Server'
  [affected_software] 'VMware ESXi'


## Part 2: Implement `build_gliner_example`

In [3]:
def find_token_span(tokenized_text, entity_text):
    entity_tokens = tokenize(entity_text)
    n = len(entity_tokens)
    for i in range(len(tokenized_text) - n + 1):
        if [t.lower() for t in tokenized_text[i:i+n]] == [t.lower() for t in entity_tokens]:
            return (i, i + n - 1)
    return None


def build_gliner_example(client, text):
    tokenized_text = tokenize(text)
    raw_output = call_llm_ner(client, text)
    entities = parse_ner_output(raw_output)
    ner_spans = []
    for entity_text, entity_type in entities:
        span = find_token_span(tokenized_text, entity_text)
        if span:
            ner_spans.append([span[0], span[1], entity_type])
    if not ner_spans:
        return None
    return {"tokenized_text": tokenized_text, "ner": ner_spans}


example = build_gliner_example(client, reports[0]['text'])
assert example is not None
n = len(example['tokenized_text'])
for s, e, label in example['ner']:
    assert 0 <= s <= e < n
    assert label in ENTITY_TYPES

print(f"✓ build_gliner_example works")
for s, e, label in example['ner']:
    print(f"  [{label}] span=[{s},{e}] → '{' '.join(example['tokenized_text'][s:e+1])}'")

✓ build_gliner_example works
  [threat_actor] span=[1,4] → 'LockBit 3 . 0'
  [vulnerability] span=[8,8] → 'CVE-2023-4966'
  [affected_software] span=[10,12] → 'Citrix NetScaler ADC'
  [affected_software] span=[36,37] → 'Windows Server'
  [affected_software] span=[39,40] → 'VMware ESXi'


## Part 3: Generate Full Dataset

In [4]:
def build_ner_dataset(client, reports):
    dataset = []
    for r in tqdm(reports, desc="Annotating NER"):
        try:
            ex = build_gliner_example(client, r['text'])
            if ex:
                dataset.append(ex)
        except Exception as e:
            print(f"  Error on report {r['id']}: {e}")
    return dataset


ner_dataset = build_ner_dataset(client, reports)

assert len(ner_dataset) >= 15
all_labels_used = set()
for ex in ner_dataset:
    n = len(ex['tokenized_text'])
    for s, e, label in ex['ner']:
        assert 0 <= s <= e < n
        assert label in ENTITY_TYPES
        all_labels_used.add(label)
assert len(all_labels_used) >= 3

save_path = os.path.normpath(os.path.join(fixtures, "..", "expected", "ner_dataset.json"))
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, 'w') as f:
    json.dump(ner_dataset, f, indent=2)

label_counts = Counter(span[2] for ex in ner_dataset for span in ex['ner'])
print(f"✓ Generated {len(ner_dataset)} NER examples")
print(f"  Entity types: {sorted(all_labels_used)}")
print(f"  Distribution: {dict(label_counts.most_common())}")
print(f"  Saved to fixtures/expected/ner_dataset.json")

Annotating NER:   0%|          | 0/20 [00:00<?, ?it/s]

Annotating NER:   5%|▌         | 1/20 [00:03<01:12,  3.81s/it]

Annotating NER:  10%|█         | 2/20 [00:04<00:39,  2.18s/it]

Annotating NER:  15%|█▌        | 3/20 [00:06<00:31,  1.87s/it]

Annotating NER:  20%|██        | 4/20 [00:07<00:22,  1.43s/it]

...

Annotating NER:  90%|█████████ | 18/20 [00:22<00:02,  1.05s/it]

Annotating NER:  95%|█████████▌| 19/20 [00:23<00:01,  1.03s/it]

Annotating NER: 100%|██████████| 20/20 [00:25<00:00,  1.44s/it]

Annotating NER: 100%|██████████| 20/20 [00:25<00:00,  1.28s/it]

✓ Generated 20 NER examples
  Entity types: ['affected_software', 'attack_technique', 'malware', 'threat_actor', 'vulnerability']
  Distribution: {'affected_software': 24, 'attack_technique': 17, 'threat_actor': 12, 'vulnerability': 10, 'malware': 7}
  Saved to fixtures/expected/ner_dataset.json
